
# Elite Dangerous Local Database
> Cached Elite Dangerous systems data

In [ ]:
#| default_exp eddb.localdb

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, logging, typing, sqlite3, os, json, gzip
import pandas as pd

from typing import Any, NamedTuple
from contextlib import contextmanager
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging
from edcompanion.eddb.readers import dbfilereader, dbfile_process
init_console_logging(__name__)

2025-12-22T06:42:07+0100 INFO	47896	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
eddb_config = configuration["EDDB"]
syslog.info(f"Loading module {__name__}, config={eddb_config}")


2025-12-22T06:42:07+0100 INFO	47896	__main__	4091137986.py	<module>	4	Loading module __main__, config=Section EDDB in C:\Users\fenke\AppData\Roaming\.config\EDTravelCompanion\settings.ini


## SQLite

### SQLiteQueryParams

In [ ]:
#| export
class SQLiteQueryParams(NamedTuple):
    as_param: typing.Callable
    append_param: typing.Callable   
    get_params: typing.Callable  


In [ ]:
#| export

def sqlite_query_params(log=None) -> SQLiteQueryParams:
    sql_params = {}

    def as_param(name:str):
        return f":{str(name)}"
   
    def append_param(name:str, value:Any):
        assert str(name) not in sql_params, f"Duplicate parameter {name}"
        assert len(sql_params) < 32766, "SQLite does not allow more then approx. 32k bound parameters"

        last_name = str(name)
        sql_params[last_name] = value
        return as_param(last_name)
    
    def get_params():
        return sql_params.copy()
    
    return SQLiteQueryParams(
        as_param=as_param,
        append_param=append_param if log is None else lambda p: log(append_param(p)),
        get_params=get_params
    )

### SQLiteConnectionInterface

In [ ]:
#| export

class SQLiteConnectionInterface(typing.NamedTuple):
    cursor: typing.Callable
    commit: typing.Callable
    rollback: typing.Callable
    close: typing.Callable
    execute: typing.Callable
    executemany: typing.Callable


In [ ]:
sys.version_info

sys.version_info(major=3, minor=12, micro=7, releaselevel='final', serial=0)

In [ ]:
#| export
def sqllite_connection_interface(
        database:str=":memory:",
    ) -> SQLiteConnectionInterface:

    vi = sys.version_info
    assert vi.major >= 3, f"Python >= 3.0 required. Found {vi.major}.{vi.minor}.{vi.micro}"

    if vi.minor > 11:   
        syslog.info("Using legacy autocommit")
        connection = sqlite3.connect(database, autocommit=sqlite3.LEGACY_TRANSACTION_CONTROL, isolation_level='DEFERRED')
    else:
        syslog.info("Using isolation_level=None")
        connection = sqlite3.connect(database, isolation_level=None)

    def close():
        syslog.info("Closing connection")
        connection.close()

    def commit():
        syslog.info("Committing on connection")
        connection.commit()

    def rollback():
        syslog.info("Rolling back connection")
        connection.rollback()
    
    def cursor():
        syslog.info("Creating cursor")
        return connection.cursor()
    
    def execute(sql:str, params:tuple|dict=()):
        return connection.execute(sql, params)
    
    def execute_many(sql:str, params:list[tuple|dict]=[]):
        return connection.executemany(sql, params)

    syslog.info("Returning connection-interface")
    return SQLiteConnectionInterface(
        cursor=cursor,
        commit=commit,
        rollback=rollback,
        close=close,
        execute=execute,
        executemany=execute_many
    )

### Context manager

In [ ]:
#| export

@contextmanager
def sqllite_connection(*args, **kwargs):
    syslog.info(f"Opening connection-interface to {args}")
    interface = sqllite_connection_interface(*args, **kwargs)
    try:
        yield interface
    finally:
        interface.close()

### Test

In [ ]:
with sqllite_connection() as ci:
    
    ci.execute("CREATE TABLE lang(name, first_appeared)")

    # This is the named style used with executemany():
    data = (
        {"name": "C", "year": 1972},
        {"name": "Fortran", "year": 1957},
        {"name": "Python", "year": 1991},
        {"name": "Go", "year": 2009},
    )
    ci.executemany("INSERT INTO lang VALUES(:name, :year)", data)
    for r in ci.execute("SELECT * FROM lang"):
        print(r)
        
    # This is the qmark style used in a SELECT query:
    params = (1972,)

    for r in ci.execute("SELECT * FROM lang WHERE first_appeared = :year", {'year':1957}):
        print(r)



2025-12-22T06:42:07+0100 INFO	47896	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ()
2025-12-22T06:42:07+0100 INFO	47896	__main__	3862603891.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-22T06:42:07+0100 INFO	47896	__main__	3862603891.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T06:42:07+0100 INFO	47896	__main__	3862603891.py	close	17	Closing connection


('C', 1972)
('Fortran', 1957)
('Python', 1991)
('Go', 2009)
('Fortran', 1957)


## ED galaxy systems database

In [ ]:
eddb_config['local_data_folder'], eddb_config['main_database']

('D:\\data\\eddb', 'eddb_systems.db')

### Systems

In [ ]:
systems_databasefile = os.path.join(eddb_config['local_data_folder'], eddb_config['main_database'])
print(systems_databasefile)

D:\data\eddb\eddb_systems.db


In [ ]:
with sqllite_connection(systems_databasefile) as conn:
    conn.execute("""
            DROP TABLE IF EXISTS systems;
        """
    )


2025-12-22T06:46:41+0100 INFO	47896	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('D:\\data\\eddb\\eddb_systems.db',)
2025-12-22T06:46:41+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-22T06:46:41+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T06:46:41+0100 INFO	47896	__main__	3552464633.py	close	17	Closing connection


In [ ]:
with sqllite_connection(systems_databasefile) as conn:
    conn.execute(f"""
    CREATE TABLE IF NOT EXISTS systems (
        id64 BIGINT NOT NULL,
        x DOUBLE PRECISION  NOT NULL,
        y DOUBLE PRECISION  NOT NULL,
        z DOUBLE PRECISION  NOT NULL,
        name TEXT NOT NULL
    );
""")


2025-12-22T06:46:42+0100 INFO	47896	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('D:\\data\\eddb\\eddb_systems.db',)
2025-12-22T06:46:42+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-22T06:46:42+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T06:46:42+0100 INFO	47896	__main__	3552464633.py	close	17	Closing connection


In [ ]:
print(f"Adding indexes ...")
with sqllite_connection(systems_databasefile) as conn:
    conn.execute(f"""
        CREATE INDEX IF NOT EXISTS systems_x_idx ON systems (x);
        CREATE INDEX IF NOT EXISTS systems_y_idx ON systems (y);
        CREATE INDEX IF NOT EXISTS systems_z_idx ON systems (z); 
        CREATE INDEX IF NOT EXISTS systems_name_idx ON systems (name);
        CREATE INDEX IF NOT EXISTS systems_id64_idx ON eddb.systems (id64)
    """)

2025-12-22T06:42:08+0100 INFO	47896	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('D:\\data\\eddb\\eddb_systems.db',)
2025-12-22T06:42:08+0100 INFO	47896	__main__	3862603891.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-22T06:42:08+0100 INFO	47896	__main__	3862603891.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T06:42:08+0100 INFO	47896	__main__	3862603891.py	close	17	Closing connection


Adding indexes ...


ProgrammingError: You can only execute one statement at a time.

In [ ]:
with sqllite_connection(systems_databasefile) as conn:

    print("Removing duplicates by id")
    conn.execute("""
            DELETE FROM systems
            WHERE rowid NOT IN (
                SELECT MIN(rowid)
                FROM systems
                GROUP BY x, z, id64
            );                 

        """
    )

    print(f"Adding unique index on system id64 ...")
    conn.execute(f"""
        DROP INDEX systems_id64_idx ;
        CREATE UNIQUE INDEX IF NOT EXISTS systems_id64_idx ON eddb.systems (id64)
    """)

#### Update systems

In [ ]:
data_dump_file = os.path.join(eddb_config['local_dumps'], 'systems_1day.json.gz')
data_dump_file

'D:\\data\\eddb\\systems_1day.json.gz'

In [ ]:
import pandas as pd, os, csv


In [ ]:
infile = os.path.join(eddb_config['local_dumps'], eddb_config['systems_1week'])
outfile = infile.replace('.json.gz', '.csv')

print(infile, outfile)


D:\data\eddb\systems_1week.json.gz D:\data\eddb\systems_1week.csv


In [ ]:
def get_coordinates_from_item(item):
    coords = item.get('coords')
    return {k:coords[k] for k in ['x','y','z']}


In [ ]:

def extract_line_item(item):
    return [item.get('id64')]+list(item.get('coords').values())+[item.get('name')]




with open(outfile, 'w', newline='') as csvfile:

    fieldnames = ['id64', 'x', 'y', 'z', 'name']
    #writer = csv.DictWriter(csvfile, fieldnames=fieldnames, extrasaction='ignore')
    writer = csv.writer(csvfile, delimiter=' ',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)

    writer.writerow(fieldnames)

    for item in dbfilereader(infile):
        writer.writerow( [item.get('id64')]+list(item.get('coords').values())+[item.get('name')])



In [ ]:
df = pd.read_csv(outfile)
df

,id64,x,y,z,name
0,1081575,-28398.03125,320.84375,12084.62500,Crooke AA-A h0
1,2194559,-6660.25000,-870.06250,-4345.59375,BD+55 191
2,2260127,-5423.00000,-38.87500,374.65625,HR 8023
3,2327759,-4724.81250,2464.31250,8063.68750,Blielia AA-A h0
4,2522295,-468.00000,-92.18750,4474.62500,Herschel 36
...,...,...,...,...,...
1027463,4491486208883752,1036.09375,-30.34375,15627.43750,Boeft GK-P a5-255
1027464,4544262766886040,1040.65625,-38.46875,15770.09375,Boeft MN-Q a18-258
1027465,4824487908117120,-10601.34375,-39.81250,20215.06250,Eorgh Prou KY-M a75-274
1027466,4824679034159976,-8816.03125,-39.50000,17949.56250,Eoch Flyuae WR-N a102-274


In [ ]:
pd.DataFrame(
    [[item.get('id64')]+list(item.get('coords').values())+[item.get('name')] for item in dbfilereader(infile)]
).to_csv(outfile, index=False)

In [ ]:
sqlite3.version

C:\Users\fenke\AppData\Local\Temp\ipykernel_47896\3532165018.py:1: DeprecationWarning: version is deprecated and will be removed in Python 3.14
  sqlite3.version


'2.6.0'

In [ ]:
def extract_line_item_to_string(item):
    return f"  ({item.get('id64')}, {', '.join(map(str, item.get('coords').values()))}, '{item.get('name')}')" #f"  ({item.get('id64')}, {','.join(item.get('coords').values())}, '{item.get('name')}')"

with sqllite_connection(systems_databasefile) as conn:
    with gzip.open(infile, 'rt') as jsonfile:
        chunksize = 8* 1024
        chunknr = 0
        query_header = f"INSERT INTO systems (id64, x, y, z, name)\nVALUES\n"

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data_values = [extract_line_item_to_string(json.loads(line.rstrip(',\n\r '))) for line in chunk if len(line) > 4]
                #print(query_header + ',\n'.join(data_values))
                conn.execute(query_header + '\n'.join(data_values) )
                #print(f"Chunk {chunknr}\n", data_values)
                #print(query_header + ',\n'.join(data_values))
                chunknr += 1
                break

            else:
                break

            print(f"Chunks processed: {chunknr}")


2025-12-22T07:42:55+0100 INFO	47896	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('D:\\data\\eddb\\eddb_systems.db',)
2025-12-22T07:42:55+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	10	Using legacy autocommit
2025-12-22T07:42:55+0100 INFO	47896	__main__	3552464633.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-22T07:42:55+0100 INFO	47896	__main__	3552464633.py	close	17	Closing connection


OperationalError: near "(": syntax error

In [ ]:
def extract_line_item(item):
    return [item.get('id64')]+list(item.get('coords').values())+[item.get('name')]

with gzip.open(infile, 'rt') as jsonfile:
    chunksize =  128 *1024 * 1024
    chunknr = 0

    while True:
        chunk = jsonfile.readlines(chunksize)
        if chunk:
            data = [
                #json.loads(line.rstrip(',\n\r '))
                extract_line_item(json.loads(line.rstrip(',\n\r ')))
                for line in chunk
                if len(line) > 4
            ]
            # pd.json_normalize(
            #     data, sep='.', errors='ignore'
            # ).to_csv(outfile.replace('.csv', f'n_{chunknr}.csv'), index=False, header=True)
            pd.DataFrame(
                data#, columns=['id64', 'x', 'y', 'z', 'name']
            ).to_csv(outfile.replace('.csv', f'_{chunknr:05d}.csv'), index=False, header=['id64', 'x', 'y', 'z', 'name'])
            # pd.DataFrame(
            #     data, columns=['id64', 'x', 'y', 'z', 'name']
            # ).to_sql( 'systems', conn, if_exists='append', index=False, method='multi')

            chunknr += 1
        else:
            break


In [ ]:
def extract_line_item(item):
    return [item.get('id64')]+list(item.get('coords').values())+[item.get('name')]


with gzip.open(infile, 'rt') as jsonfile:
    chunksize = 128 * 1024 * 1024
    chunknr = 0

    while True:
        chunk = jsonfile.readlines(chunksize)
        if chunk:
            # outfile.replace('.csv', f'_{chunknr:05d}.csv')
            with open(outfile.replace('.csv', f'_{chunknr:05d}.csv'), 'w', newline='') as csvfile:
                fieldnames = ['id64', 'x', 'y', 'z', 'name']
                writer = csv.writer(csvfile, delimiter=' ',
                                        quotechar='|', quoting=csv.QUOTE_MINIMAL)

                writer.writerow(fieldnames)

                for line in chunk:
                    if len(line) > 4:
                        item = json.loads(line.rstrip(',\n\r '))
                        writer.writerow( [item.get('id64')]+list(item.get('coords').values())+[item.get('name')])
               

            chunknr += 1

        else:
            break


In [ ]:
def get_coordinates_from_item(item):
    coords = item.get('coords')
    return {k:coords[k] for k in ['x','y','z']}

conn = sqllite_connection_interface(systems_databasefile)
query = """
    INSERT INTO systems (id64, x, y, z, name) 
    VALUES (:id,:x,:y,:z,:name) 
    ON CONFLICT DO NOTHING

"""


def process_data(datachunk):

    return conn.executemany(query, [
                dict(id=item.get("id64"), **get_coordinates_from_item(item), name=item.get('name'))
                for item in datachunk
            ]
    )

dbfile_process(
    data_dump_file,
    process_data,
)

### Star types

In [ ]:
main_sequence = {s:c for s, c in zip('OBAFGKMN', range(9))}
#print(json.dumps(classifications, indent=2))


In [ ]:

star_types={}
star_types_file = os.path.join(eddb_config['local_data_folder'], 'star_types.json')
print(f"File with star types: {star_types_file}")


File with star types: /home/fenke/repos/EDCompanion/data/star_types.json


In [ ]:

if not os.path.exists(star_types_file):
    print("Extracting star types")
    skipped = 0
    updated = 0

    for item in dbfilereader(os.path.join(eddb_config['local_dumps'], eddb_config['systems_1month'])):
        main_star = item.get('mainStar')

        if main_star and not star_types.get(main_star):
            print(f"updating with {main_star}")
            updated += 1
            star_types[main_star] = len(star_types)
        else:
            skipped += 1


    print(f"Updated: {updated}, skipped {skipped}")


    main_star_types = {}
    for name in set(star_types.keys()):
        first, *rest = name.split(' ')
        if len(first) == 1 and first in main_sequence:
            main_star_types[name] = len(main_star_types)

    for name in star_types:
        if name not in main_star_types:
            main_star_types[name] = len(main_star_types)

    try:
        with open(star_types_file,'wt') as jsonfile:
            json.dump(main_star_types, jsonfile, indent=3)
    except:
        pass

    star_types = main_star_types.copy()
    
else:
    with open(star_types_file,'rt') as jsonfile:
        star_types.update(json.load(jsonfile))

#print(json.dumps(main_star_types, indent=2))


In [ ]:
main_sequence

{'O': 0, 'B': 1, 'A': 2, 'F': 3, 'G': 4, 'K': 5, 'M': 6, 'N': 7}

In [ ]:
os.path.join(eddb_config['local_dumps'], eddb_config['systems_1week'])

'/home/fenke/repos/EDCompanion/data/systems_1week.json.gz'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()